In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import mlflow
from functools import wraps
from statsmodels.tsa.stattools import adfuller, kpss
import os

# Main Flow

```text
Start
│
├── 1. Problem Definition
│
├── 2. Data Analysis & Cleaning
│
├── 3. Feature Engineering
│
├── 4. Validation Strategy
│
├── 5. Feature Selection
│
├── 6. Model Research
│
├── 7. Risk & Time Estimation
│
├── 8. Candidate Model Selection
│
├── 9. Model-specific Data Pipelines
│
├── 10. Fast Hyperparameter Tuning
│
├── 11. Error Analysis
│
├── 12. Benchmark Candidate Models
│
├── 13. Final Model Selection
│
├── 14. Full Hyperparameter Tuning
│
├── 15. Final Evaluation
│
└── End
```

# Problem Definition

## Project Title

**Hierarchical Probabilistic Sales Forecasting using the FreshRetailNet-50K Dataset**

---

# Objective

The objective of this project is to develop a **Hierarchical Probabilistic Time Series Forecasting** system capable of predicting future product sales across multiple levels of the retail hierarchy while quantifying the uncertainty associated with each prediction.

The forecasting system should generate coherent forecasts across hierarchical levels, ensuring that predictions remain consistent between aggregated and disaggregated nodes. In addition to point forecasts, the model should estimate prediction intervals to support inventory management, demand planning, and business decision-making.

---

# Problem Type

- Supervised Learning
- Time Series Forecasting
- Hierarchical Forecasting
- Probabilistic Forecasting

---

# Prediction Target

**Target Variable**

| Feature | Description |
|----------|-------------|
| `sale_amount` | Normalized daily sales amount of each product. The model aims to forecast future values of this variable together with the associated uncertainty. |

---

# Input Features

| Feature | توضیح |
|----------|-------|
| `city_id` | شناسه شهر محل فروشگاه |
| `store_id` | شناسه فروشگاه |
| `management_group_id` | شناسه گروه مدیریتی |
| `first_category_id` | شناسه دسته‌بندی سطح اول محصول |
| `second_category_id` | شناسه دسته‌بندی سطح دوم محصول |
| `third_category_id` | شناسه دسته‌بندی سطح سوم محصول |
| `product_id` | شناسه محصول |
| `dt` | تاریخ رکورد |
| `hours_sale` | بردار فروش ساعتی محصول در طول روز |
| `stock_hour6_22_cnt` | تعداد ساعات ناموجود بودن کالا بین ساعت ۶ تا ۲۲ |
| `hours_stock_status` | بردار وضعیت موجودی کالا در ساعات مختلف |
| `discount` | نرخ تخفیف اعمال‌شده (۱ به معنی بدون تخفیف) |
| `holiday_flag` | نشان‌دهنده تعطیل بودن روز |
| `activity_flag` | نشان‌دهنده وجود کمپین یا فعالیت فروش |
| `precpt` | میزان بارندگی روز |
| `avg_temperature` | میانگین دمای روز |
| `avg_humidity` | میانگین رطوبت روز |
| `avg_wind_level` | میانگین شدت باد |

---

# Dataset Characteristics

The dataset contains historical retail sales records together with multiple groups of explanatory variables:

- Product hierarchy
- Store hierarchy
- Historical sales
- Inventory information
- Promotion information
- Calendar information
- Weather information

## Organizational Hierarchy

```text
City
└── Store
```

## Product Hierarchy

```text
Management Group
└── First Category
    └── Second Category
        └── Third Category
            └── Product
```

The forecasting task is designed to exploit these hierarchical relationships in order to produce coherent forecasts across different aggregation levels.

---

# Expected Output

The final forecasting system should generate:

- Point Forecasts
- Prediction Intervals
- Forecast Uncertainty Estimation

The output should be available across the required hierarchical levels while maintaining coherence between parent and child nodes.

---

# Success Criteria

The final model should:

- Produce accurate point forecasts.
- Generate reliable prediction intervals.
- Maintain hierarchical consistency across all forecast levels.
- Generalize well to unseen future observations.
- Be computationally efficient for both training and inference.
- Be suitable for deployment in a retail demand forecasting pipeline.

---

# Constraints

- Future information must never leak into the training process.
- All validation procedures must preserve temporal ordering.
- Hierarchical relationships must be respected throughout the forecasting pipeline.
- The entire workflow should be fully reproducible.
- All experiments will be tracked using MLflow.

## Mlflow base infrastructure

In [2]:
def run_step(step_name):

    def decorator(func):

        @wraps(func)
        def wrapper(*args, **kwargs):

            with mlflow.start_run(
                run_name=step_name,
                nested=False
            ):

                result = func(*args, **kwargs)

            return result

        return wrapper

    return decorator

In [3]:
db_path = os.path.join(os.getcwd(), "mlflow.db")

mlflow.set_tracking_uri(f"sqlite:///{db_path}")

mlflow.set_experiment("Sales_Forecasting_FreshRetailNet-50K Dataset")

<Experiment: artifact_location='file:///e:/FreshRetailNet/mlruns/1', creation_time=1784314327891, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784314327891, lifecycle_stage='active', name='Sales_Forecasting_FreshRetailNet-50K Dataset', tags={}, trace_location=None, workspace='default'>

# Step 2 - Data Analysis & Cleaning
```text

├── 2.1 Dataset Overview
├── 2.2 Data Quality Analysis
├── 2.3 Hierarchy Analysis
├── 2.4 Temporal Analysis
├── 2.5 Target Analysis
├── 2.6 Entity Analysis
├── 2.7 Covariate Analysis
└── 2.8 Summary & Cleaning Decisions
```

```text
هر تحلیل (Analysis) باید دقیقاً پنج خروجی داشته باشد.

Statistics
Visualization (در صورت نیاز)
Interpretation
Decision
MLflow Logging
```

In [4]:
dataset = load_dataset("Dingdong-Inc/FreshRetailNet-50K",cache_dir="data")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 4500000
    })
    eval: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 350000
    })
})


In [8]:
stores = list(set(dataset["train"]["store_id"]))

In [13]:
max(stores)

897

In [19]:
import random

random.seed(42)

SELECTED_STORES = random.sample(
    sorted(set(dataset["train"]["store_id"])),
    k=10
)

print(SELECTED_STORES)

[654, 114, 25, 759, 281, 250, 228, 142, 754, 104]


In [20]:
[654, 114, 25, 759, 281, 250, 228, 142, 754, 104]

[654, 114, 25, 759, 281, 250, 228, 142, 754, 104]

In [21]:
dataset_filterd = dataset.filter(
    lambda x: x["store_id"] in SELECTED_STORES
)

Filter:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/350000 [00:00<?, ? examples/s]

In [22]:
dataset_filterd

DatasetDict({
    train: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 47340
    })
    eval: Dataset({
        features: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level'],
        num_rows: 3682
    })
})

In [25]:
train = pd.DataFrame(dataset_filterd["train"])
test = pd.DataFrame(dataset_filterd["eval"])

In [30]:
train.to_csv("csv_data/train.csv",index=False)
test.to_csv("csv_data/test.csv",index=False)

In [4]:
train = pd.read_csv("csv_data/train.csv")
#test = pd.read_csv("csv_data/test.csv")

In [5]:
train

,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,dt,sale_amount,hours_sale,stock_hour6_22_cnt,hours_stock_status,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,0,104,0,5,7,67,37,2024-03-28,0.3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",10,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, ...",1.000,0,0,1.6999,15.90,73.49,1.36
1,0,104,0,5,7,67,37,2024-03-29,0.3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.000,0,0,3.0183,15.52,76.57,1.53
2,0,104,0,5,7,67,37,2024-03-30,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",16,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1.000,1,0,2.0944,16.40,76.60,1.69
3,0,104,0,5,7,67,37,2024-03-31,0.3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",6,"[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.000,1,0,1.5622,16.96,77.63,1.59
4,0,104,0,5,7,67,37,2024-04-01,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",16,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",1.000,0,0,3.3341,15.79,77.86,1.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47335,13,759,6,21,64,123,272,2024-06-21,1.2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.745,0,1,7.4353,28.97,81.57,2.04
47336,13,759,6,21,64,123,272,2024-06-22,1.6,"[0.0, 0.0, 0.0, 0.1, 0.0, 0.1, 0.1, 0.0, 0.4, ...",0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.760,1,1,4.6834,29.32,81.44,1.77
47337,13,759,6,21,64,123,272,2024-06-23,1.9,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.2, 0.2, ...",0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.726,1,1,4.1763,29.28,80.12,1.90
47338,13,759,6,21,64,123,272,2024-06-24,0.3,"[0.2, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.0, ...",14,"[0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, ...",0.739,0,1,3.8238,29.38,80.70,2.04


In [6]:
train.groupby(by="store_id").apply(lambda x: sum(x["sale_amount"]))

store_id
25     3825.259
104    5741.780
114    1103.510
142    5718.550
228    3828.580
250    7919.159
281    5673.750
654    4591.160
754    1480.535
759    1183.350
dtype: float64